# Lab 5

Finish a function in `lab5.py`, then run its cell. Two ideas: a sharper estimate than last week's, and improving a plan you already have instead of searching for one.

In [ ]:
import time, random
from lab2 import load, successors, goal_reached, Problem
from lab3 import bfs, dfs, ucs, plan_cost
from lab4 import greedy, astar, goal_count, h_add, best_first
from lab5 import *

## A relaxed plan, not a relaxed sum

`h_add` adds up the relaxed cost of every goal fact, so work that two goals share is paid twice. `h_ff` builds the relaxed planning graph (`relaxed_layers`, given), extracts one relaxed plan backwards, and counts its actions once.

In [ ]:
d, p2, a2 = load("domain.pddl", "problem.pddl")
facts, layers = relaxed_layers(p2.init, p2.goal, a2)
print("fact layers:", [len(f) for f in facts], " action layers:", [len(l) for l in layers])
print("goal_count", goal_count(p2.init, p2.goal, a2), " h_add", h_add(p2.init, p2.goal, a2), " h_ff", h_ff(p2.init, p2.goal, a2), " truth", ucs(p2, a2).cost)

Three layers reach the goal. `h_add` says 10, `h_ff` says 7; the true cheapest plan costs 12 and needs 9 actions. The relaxed plan is short because nothing in it has to be put down or walked back.

In [ ]:
d, p4, a4 = load("domain.pddl", "p4.pddl")
d, p6, a6 = load("domain.pddl", "p6.pddl")
for name, h in [("goal_count", goal_count), ("h_add", h_add), ("h_ff", h_ff)]:
    t = time.time(); r = greedy(p4, a4, h); print(f"four items, greedy with {name:10s} {r}  ({time.time() - t:.2f}s)")
for name, h in [("goal_count", goal_count), ("h_add", h_add), ("h_ff", h_ff)]:
    t = time.time(); r = greedy(p6, a6, h); print(f"six items,  greedy with {name:10s} {r}  ({time.time() - t:.2f}s)")

On six items `h_ff` expands 20 states where `goal_count` needs 79 and `h_add` 27, and returns the cost-24 plan that `h_add` missed. Now the ten-item problem:

In [ ]:
d, p10, a10 = load("domain.pddl", "p10.pddl")
t = time.time(); print("ten items, greedy with goal_count:", greedy(p10, a10, goal_count, 3_000), f"({time.time() - t:.1f}s)")
t = time.time(); print("ten items, greedy with h_ff:      ", greedy(p10, a10, h_ff, 3_000), f"({time.time() - t:.1f}s)")

The crude estimate solves it in 144 expansions; the sharp one finds nothing in 3,000. The goal has a stack in it, and a relaxation that never deletes cannot see that a block stacked too early must come off again. The better heuristic is not better everywhere, and knowing where it fails is worth more than its average.

## Weighted A*: the dial between A* and greedy

Priority `g + w * h`. With `w = 1` it is A*; the larger `w`, the more it trusts the estimate. Each run is allowed 5,000 expansions.

In [ ]:
print("greedy, h_ff:", greedy(p6, a6, h_ff))
for w in (1.0, 1.5, 2.0, 5.0):
    t = time.time(); r = weighted_astar(p6, a6, h_ff, w, 5_000)
    print(f"w = {w}: {r}  ({time.time() - t:.1f}s)")

Six items with `h_ff`. `w = 1` runs out of its 5,000 expansions with nothing. `w = 1.5` finds cost 23 for about 3,000 expansions; `w = 2` finds 24 for 400; `w = 5` finds 24 for 24, like greedy. Nobody promised the 23 was optimal, but it is cheaper than anything greedy found, and A* could not afford to find it at all.

## Local search in plan space: pruning

A plan is a state too. The simplest move is deleting an action; `prune_plan` deletes any single action, or any pair, that leaves the plan valid, until nothing can go.

In [ ]:
for name, pr, ac in [("two animals", p2, a2), ("six items", p6, a6)]:
    plan = dfs(pr, ac).plan
    t = time.time(); pruned = prune_plan(pr, ac, plan)
    print(f"{name}: depth-first {len(plan)} actions, cost {plan_cost(plan)}  ->  pruned {len(pruned)} actions, cost {plan_cost(pruned)}  ({time.time() - t:.1f}s)")

Depth-first's 27-action wander on the two animals prunes to 9 actions, cost 12: the optimal plan, found by deleting. On six items 30 actions at cost 46 become 21 at cost 34, and that is where deleting stops.

## Local search in plan space: annealing

Deletion cannot reorder anything. `anneal` takes any state, any neighbour function and any score. First the docstring's toy, then plans: a neighbour keeps a random prefix of the plan and re-plans the rest with a noisy greedy search.

In [ ]:
print("toy:", anneal(10, lambda x: x + random.choice((-1, 1)), lambda x: abs(x - 3), seconds=0.5))
by_name = {a.name: a for a in a6}
def after(prefix):
    s = p6.init
    for n in prefix:
        s = (s - by_name[n].delete) | by_name[n].add
    return s
rng = random.Random(0)
def neighbour(plan):
    cut = rng.randrange(len(plan))
    sub = Problem(p6.name, p6.domain, p6.objects, after(plan[:cut]), p6.goal)
    r = best_first(sub, a6, lambda g, s: goal_count(s, p6.goal, a6) + 0.9 * rng.random(), 2000)
    return plan[:cut] + r.plan if r.plan is not None else plan
start = prune_plan(p6, a6, dfs(p6, a6).plan)
t = time.time(); best = anneal(start, neighbour, plan_cost, seconds=15, temperature=2.0)
print(f"six items: pruned depth-first cost {plan_cost(start)}  ->  annealed cost {plan_cost(best)}, {len(best)} actions, valid {goal_reached(after(best), p6.goal)}  ({time.time() - t:.0f}s)")

From 34 to the mid-twenties in fifteen seconds, with no frontier and no memory beyond one plan. Weighted A* found 23 above; nobody in this notebook knows the optimum. That is local search: a better answer, never a proof.

Next week: nothing to plan for; agents that play against each other.